# Phase 5: LightGBM Model Training & Macro $F_{0.5}$ Calibration
### **Amazon ML Challenge 2026 — Business Entity Resolution**

This notebook implements the complete **Phase 5** training and calibration pipeline:
1. **Install Dependencies** (`polars`, `lightgbm`, `rapidfuzz`, `pyarrow`)
2. **Load 27-Dimensional Pairwise Feature Dataset** (`train_features.parquet`)
3. **Train Precision-Optimized LightGBM Classifier** with GBDT trees
4. **Feature Importance Analysis** (Split Gain)
5. **Low-Memory Stratified Validation Setup** (Streamlined attribute caching)
6. **Batch Validation Scoring & Macro $F_{0.5}$ Decision Threshold Optimization** (Sweeping $\theta \in [0.30, 0.95]$ with singleton defense)
7. **Export Model Artifacts** (`lgbm_entity_resolver.txt` & `model_config.json`) & S3 Backup

## 1. Install Dependencies (SageMaker / Cloud Kernel)

In [ ]:
# Install required packages in the active SageMaker Jupyter kernel
%pip install -q polars lightgbm rapidfuzz pyarrow matplotlib

## 2. Setup, Environment & Imports

In [ ]:
import os
import sys
import time
import json
import gc
from pathlib import Path
import numpy as np
import polars as pl
import lightgbm as lgb
import matplotlib.pyplot as plt

# Robust auto-detection of repository and src path in SageMaker or Local
cur = Path.cwd().resolve()
for p in [cur, cur.parent, cur.parent.parent, cur.parent.parent.parent]:
    if (p / "src").exists():
        sys.path.insert(0, str(p))
        break
    if (p / "code" / "business_entity_resolution" / "src").exists():
        sys.path.insert(0, str(p / "code" / "business_entity_resolution"))
        break

from src.config import PROCESSED_DIR, MODELS_DIR
from src.feature_engineering import FeatureExtractor, FEATURE_NAMES
from src.evaluator import evaluate_entity_resolution

MODELS_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print(f"Processed Data Dir:  {PROCESSED_DIR}")
print(f"Models Directory:    {MODELS_DIR}")
print("✓ Imports loaded successfully!")

## 3. Sync Processed Datasets from S3 (If Running on SageMaker)

In [ ]:
train_features_path = PROCESSED_DIR / "train_features.parquet"
cand_pairs_path = PROCESSED_DIR / "val_candidate_pairs.parquet"
val_gt_path = PROCESSED_DIR / "val_ground_truth.parquet"
val_s1_path = PROCESSED_DIR / "val_source1_cleaned.parquet"
train_s2_path = PROCESSED_DIR / "train_source2_cleaned.parquet"
train_s3_path = PROCESSED_DIR / "train_source3_cleaned.parquet"

S3_PROCESSED_URI = "s3://amazon-sagemaker-720682844180-us-east-1-axi666gcrhqjon/shared/dataset/processed/"

# Auto-download from S3 if train_features.parquet is not found locally
if not train_features_path.exists():
    print(f"Pulling processed datasets from S3: {S3_PROCESSED_URI}...")
    !aws s3 sync {S3_PROCESSED_URI} {PROCESSED_DIR}/
else:
    print(f"✓ train_features.parquet already present ({train_features_path.stat().st_size / (1024*1024):.2f} MB).")

## 4. Load & Inspect 27-Dimensional Pairwise Feature Matrix

In [ ]:
print("=" * 70)
print("LOADING PAIRWISE TRAINING FEATURES")
print("=" * 70)

t0 = time.time()
df_train = pl.read_parquet(train_features_path)
print(f"Loaded {len(df_train):,} training pairs in {time.time() - t0:.2f}s.\n")

X_train = df_train.select(FEATURE_NAMES).to_numpy().astype(np.float32)
y_train = df_train["label"].to_numpy().astype(np.int32)

n_pos = int((y_train == 1).sum())
n_neg = int((y_train == 0).sum())

print(f"• Feature Matrix Shape:  {X_train.shape}")
print(f"• True Matches (y=1):    {n_pos:,} ({n_pos / len(y_train) * 100:.1f}%)")
print(f"• Hard Negatives (y=0):  {n_neg:,} ({n_neg / len(y_train) * 100:.1f}%)")
print(f"• Negative-to-Positive:  {n_neg / n_pos:.1f} : 1.0")

# Clean up DataFrame from RAM
del df_train
gc.collect()

## 5. Train LightGBM Pairwise GBDT Classifier

We train a high-precision gradient boosted decision tree model with:
- `learning_rate = 0.05`
- `num_leaves = 63`
- `max_depth = 8`
- `feature_fraction = 0.85`, `bagging_fraction = 0.85`
- `num_boost_round = 300`

In [ ]:
print("=" * 70)
print("TRAINING LIGHTGBM MODEL")
print("=" * 70)

train_data = lgb.Dataset(X_train, label=y_train, feature_name=FEATURE_NAMES)

params = {
    "objective": "binary",
    "metric": "binary_logloss",
    "boosting_type": "gbdt",
    "learning_rate": 0.05,
    "num_leaves": 63,
    "max_depth": 8,
    "min_child_samples": 30,
    "feature_fraction": 0.85,
    "bagging_fraction": 0.85,
    "bagging_freq": 1,
    "verbosity": 1,
    "n_jobs": -1,
    "seed": 42,
}

t_train = time.time()
model = lgb.train(
    params,
    train_data,
    num_boost_round=300,
)
train_elapsed = time.time() - t_train
print(f"\n✓ Model training completed in {train_elapsed:.2f}s!")

# Free training data from RAM
del X_train, y_train, train_data
gc.collect()

## 6. Feature Importance Analysis (Split Gain)

In [ ]:
importance = model.feature_importance(importance_type="gain")
sorted_idx = np.argsort(importance)[::-1]

print("Top 15 Most Predictive Features (Split Gain):")
for rank, idx in enumerate(sorted_idx[:15], 1):
    print(f"  {rank:2d}. {FEATURE_NAMES[idx]:<28} Gain: {importance[idx]:,.1f}")

# Plot Feature Importances
plt.figure(figsize=(10, 6))
top_names = [FEATURE_NAMES[i] for i in sorted_idx[:15]][::-1]
top_gains = [importance[i] for i in sorted_idx[:15]][::-1]
plt.barh(top_names, top_gains, color="#2563eb")
plt.xlabel("Split Gain (Importance)")
plt.title("Top 15 Feature Importances — LightGBM Entity Resolver")
plt.tight_layout()
plt.show()

## 7. Low-Memory Validation Set Setup & Sequential Attribute Caching

We calibrate the decision threshold $\theta^*$ against the validation ground truth.
To keep memory usage strictly under **20% RAM**, we load datasets sequentially with column pruning and garbage collection.

In [ ]:
print("=" * 70)
print("SETTING UP LOW-MEMORY VALIDATION CALIBRATION SET")
print("=" * 70)

t_val_start = time.time()
val_cand_df = pl.read_parquet(cand_pairs_path)
val_gt = pl.read_parquet(val_gt_path)

val_gt_map = {}
for row in val_gt.iter_rows():
    s1_id, matches = row[0], row[1]
    if matches and str(matches).strip():
        val_gt_map[s1_id] = set(str(matches).strip().split(","))
    else:
        val_gt_map[s1_id] = set()

# Calibration sample (20,000 validation entities)
calib_size = min(20000, len(val_cand_df))
print(f"Calibrating threshold on {calib_size:,} validation entities...")

calib_cand_df = val_cand_df.head(calib_size)
calib_s1_ids = calib_cand_df["source1_entity_id"].to_list()
calib_tgt_ids = set()
for cands_str in calib_cand_df["candidate_entity_ids"].to_list():
    if cands_str:
        for tgt in str(cands_str).split(","):
            tgt = tgt.strip()
            if tgt:
                calib_tgt_ids.add(tgt)

needed_s1_set = set(calib_s1_ids)
needed_tgt_set = calib_tgt_ids

cols = ["entity_id", "name_clean", "addr_clean", "name_tokens", "postal_digits"]
extractor = FeatureExtractor()

# 1. Register S1 entities
val_s1 = pl.read_parquet(val_s1_path, columns=cols).filter(pl.col("entity_id").is_in(needed_s1_set))
extractor.register_dataset(val_s1)
del val_s1
gc.collect()

# 2. Register S2 target candidates
train_s2 = pl.read_parquet(train_s2_path, columns=cols).filter(pl.col("entity_id").is_in(needed_tgt_set))
extractor.register_dataset(train_s2)
del train_s2
gc.collect()

# 3. Register S3 target candidates
train_s3 = pl.read_parquet(train_s3_path, columns=cols).filter(pl.col("entity_id").is_in(needed_tgt_set))
extractor.register_dataset(train_s3)
del train_s3
gc.collect()

print(f"✓ Cached {len(extractor.entity_lookup):,} entities in fast attribute lookup in {time.time() - t_val_start:.2f}s!")

## 8. Batch Validation Scoring & Threshold $\theta$ Sweep

We score candidate pairs in chunks of $50,000$ to maintain ultra-fast inference with minimal memory footprint.
Then, we sweep $\theta \in [0.30, 0.95]$ to identify the optimal decision threshold $\theta^*$ that maximizes exact **Macro $F_{0.5}$** with **singleton defense**.

In [ ]:
print("=" * 70)
print("VALIDATION INFERENCE & MACRO F0.5 THRESHOLD CALIBRATION")
print("=" * 70)

val_cands_list = calib_cand_df["candidate_entity_ids"].to_list()

flat_pairs = []
pair_to_s1 = []
pair_to_tgt = []

for s1_id, cands_str in zip(calib_s1_ids, val_cands_list):
    if not cands_str:
        continue
    for tgt_id in str(cands_str).split(","):
        tgt_id = tgt_id.strip()
        if tgt_id:
            flat_pairs.append((s1_id, tgt_id))
            pair_to_s1.append(s1_id)
            pair_to_tgt.append(tgt_id)

print(f"Extracting features and scoring {len(flat_pairs):,} validation candidate pairs...")
t_feat = time.time()

# Batch inference
probs = []
batch_size = 50000
for i in range(0, len(flat_pairs), batch_size):
    batch_pairs = flat_pairs[i:i + batch_size]
    X_batch, _ = extractor.extract_features_for_pairs(batch_pairs)
    p_batch = model.predict(X_batch)
    probs.extend(p_batch)
    del X_batch, p_batch
    gc.collect()

print(f"Inference completed in {time.time() - t_feat:.2f}s ({len(flat_pairs)/(time.time() - t_feat):,.0f} pairs/s).\n")

# Group predicted candidate probabilities by S1 entity
s1_to_scored_cands = {s1_id: [] for s1_id in calib_s1_ids}
for s1_id, tgt_id, p in zip(pair_to_s1, pair_to_tgt, probs):
    s1_to_scored_cands[s1_id].append((tgt_id, float(p)))

calib_gt_map = {s1_id: val_gt_map.get(s1_id, set()) for s1_id in calib_s1_ids}

# Threshold Sweep
thresholds = np.arange(0.30, 0.96, 0.05)
best_f05 = -1.0
best_thresh = 0.50
best_metrics = {}

f05_scores = []
precisions = []
recalls = []

print(f"{'Threshold θ':<12} | {'Macro F0.5':<12} | {'Precision':<12} | {'Recall':<12}")
print("-" * 55)

for th in thresholds:
    preds = {}
    for s1_id, scored_list in s1_to_scored_cands.items():
        matched = {tgt_id for tgt_id, p in scored_list if p >= th}
        preds[s1_id] = matched

    eval_res = evaluate_entity_resolution(calib_gt_map, preds, beta=0.5)
    f05 = eval_res["macro_f05"]
    p = eval_res["macro_precision"]
    r = eval_res["macro_recall"]
    
    f05_scores.append(f05)
    precisions.append(p)
    recalls.append(r)
    
    print(f"  θ = {th:.2f}     | {f05:.4f}       | {p:.4f}       | {r:.4f}")

    if f05 > best_f05:
        best_f05 = f05
        best_thresh = float(th)
        best_metrics = eval_res

print("=" * 70)
print(f"OPTIMAL DECISION THRESHOLD: θ* = {best_thresh:.2f}")
print(f"  • Best Macro F0.5:   {best_f05:.4f}")
print(f"  • Macro Precision:   {best_metrics['macro_precision']:.4f}")
print(f"  • Macro Recall:      {best_metrics['macro_recall']:.4f}")
print("=" * 70)

## 9. Visualize Threshold Calibration Curve

In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(thresholds, f05_scores, marker="o", color="#2563eb", label="Macro F0.5 (Competition Metric)", linewidth=2.5)
plt.plot(thresholds, precisions, marker="s", color="#16a34a", linestyle="--", label="Macro Precision")
plt.plot(thresholds, recalls, marker="^", color="#dc2626", linestyle="--", label="Macro Recall")
plt.axvline(best_thresh, color="#f59e0b", linestyle=":", linewidth=2, label=f"Optimal θ* = {best_thresh:.2f} (F0.5={best_f05:.4f})")
plt.xlabel("Decision Threshold θ")
plt.ylabel("Validation Score")
plt.title("Macro F0.5 Threshold Optimization & Calibration Curve")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 10. Save Model Artifacts & Backup to S3

In [ ]:
model_path = MODELS_DIR / "lgbm_entity_resolver.txt"
meta_path = MODELS_DIR / "model_config.json"

# 1. Save LightGBM booster model
model.save_model(str(model_path))

# 2. Save metadata and calibrated threshold config
config_payload = {
    "optimal_threshold": best_thresh,
    "best_macro_f05": best_f05,
    "best_precision": best_metrics["macro_precision"],
    "best_recall": best_metrics["macro_recall"],
    "feature_names": FEATURE_NAMES,
    "num_boost_round": 300,
    "calib_entities": calib_size,
}

with open(meta_path, "w", encoding="utf-8") as f:
    json.dump(config_payload, f, indent=2)

print(f"✓ Model booster saved to:    {model_path}")
print(f"✓ Model config saved to:     {meta_path}")

# 3. Backup to S3
!aws s3 cp {model_path} s3://amazon-sagemaker-720682844180-us-east-1-axi666gcrhqjon/shared/models/lgbm_entity_resolver.txt
!aws s3 cp {meta_path} s3://amazon-sagemaker-720682844180-us-east-1-axi666gcrhqjon/shared/models/model_config.json
print("✓ Model artifacts successfully backed up to AWS S3!")